In [ ]:
# ============================================================
# Section 1 — Diagnostics + Correct forward citations
# Uses GDATE if available; otherwise uses year with lag>=0
# ============================================================


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Mount google drive
from google.colab import drive
drive.mount("/content/drive")

PATENTS_CSV_PATH = "/content/drive/MyDrive/Code LateX Templates/apat63_99.csv"
CITES_CSV_PATH   = "/content/drive/MyDrive/Code LateX Templates/cite75_99.csv"

In [ ]:
# we read the csv file and create the variable cols corresponding to each header or column 0. We then load selected columns (usecols). We append dates to columns headers where present.
def load_patents_dynamic(patents_csv_path: str) -> pd.DataFrame:
   """
   Load patents CSV and include GDATE if available.
   """
   cols = pd.read_csv(patents_csv_path, nrows=0).columns.tolist()
   base_cols = ["PATENT", "GYEAR", "APPYEAR", "COUNTRY", "POSTATE", "ASSIGNEE", "ASSCODE", "NCLASS", "CAT", "SUBCAT"]
   usecols = [c for c in base_cols if c in cols]
   if "GDATE" in cols:
       usecols.append("GDATE")




   df = pd.read_csv(patents_csv_path, usecols=usecols, low_memory=False)


   # numeric coercions
	# we use def…coerce to keep as NaN, values that cannot be transformed in numeric values, with NaN behaving as an integer type (astype(In64)).
# we then do the same for GDATE when present.
   for c in ["PATENT", "GYEAR", "APPYEAR", "ASSIGNEE", "ASSCODE"]:
       if c in df.columns:
           df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
   if "GDATE" in df.columns:
       df["GDATE"] = pd.to_numeric(df["GDATE"], errors="coerce").astype("Int64")


   # strings
	# same as earlier but in forcing storage as strings for relevant categories, ie. variables expected to behave as strings like country or CAT (category).
   for c in ["COUNTRY", "POSTATE", "CAT", "SUBCAT"]:
       if c in df.columns:
           df[c] = df[c].astype("string")


   return df

In [ ]:
# same as earlier but with the other patent citations file, with non-numeric entries becoming NaN and dropping missing variables rows.
def load_citations(cites_csv_path: str) -> pd.DataFrame:
   df = pd.read_csv(cites_csv_path, usecols=["CITING", "CITED"], low_memory=False)
   df["CITING"] = pd.to_numeric(df["CITING"], errors="coerce").astype("Int64")
   df["CITED"]  = pd.to_numeric(df["CITED"], errors="coerce").astype("Int64")
   df = df.dropna(subset=["CITING", "CITED"]).copy()
   return df


# building a citation window to consider only citations building up during a 5 year frame, to avoid skewing results of older patents having more time than younger ones to accumulate citations. This will logically truncate our results 5 years before the end of the dataset (ending in 1994 instead of 1999).
def construct_forward_citations_fixed_window(
   patents: pd.DataFrame,
   citations: pd.DataFrame,
   focal_start_year: int = 1975,
   focal_end_year: int = 1994,
   window_years: int = 5
) -> pd.DataFrame:
   """
   Forward citations within a fixed 5-year window.


   - If GDATE exists: use week-based lag and keep 0..(window_years*52+3)
   - Else: use year-based lag and keep 0..window_years


   Returns focal patents with fwd_cites_5y and self_cites_5y.
   """
   # Minimal metadata
   meta_cols = ["PATENT", "GYEAR", "ASSIGNEE", "ASSCODE", "NCLASS", "CAT", "SUBCAT"]
   if "GDATE" in patents.columns:
       meta_cols.append("GDATE")


   meta = patents[meta_cols].copy()


   # focal patents (complete 5y window in cite75_99 through 1999 => end 1994)
   focal = meta[(meta["GYEAR"] >= focal_start_year) & (meta["GYEAR"] <= focal_end_year)].copy()


   # Merge citations onto focal as "cited" metadata (inner join filters to focal targets, ie. discards citing to a patent outside the focal group, keeps citations to a focal patent), including GDATE if available, then drop missing time fields.
   cited_merge = focal[["PATENT", "GYEAR", "ASSIGNEE", "ASSCODE"] + (["GDATE"] if "GDATE" in focal.columns else [])].rename(columns={
       "PATENT": "CITED",
       "GYEAR": "cited_year",
       "ASSIGNEE": "cited_assignee",
       "ASSCODE": "cited_asscode",
       **({"GDATE": "cited_gdate"} if "GDATE" in focal.columns else {})
   })
   cit = citations.merge(cited_merge, on="CITED", how="inner")


   # Merge citing metadata as table.
   citing_merge = meta[["PATENT", "GYEAR", "ASSIGNEE", "ASSCODE"] + (["GDATE"] if "GDATE" in meta.columns else [])].rename(columns={
       "PATENT": "CITING",
       "GYEAR": "citing_year",
       "ASSIGNEE": "citing_assignee",
       "ASSCODE": "citing_asscode",
       **({"GDATE": "citing_gdate"} if "GDATE" in meta.columns else {})
   })
   cit = cit.merge(citing_merge, on="CITING", how="left")


   # window filter, with missing values becoming NaN
   if "GDATE" in meta.columns:
       cit = cit.dropna(subset=["citing_gdate", "cited_gdate"]).copy()
       # weeks in ~5 years with small slack added for leap-week issues and year-length irregularities + “citation lag” non-negative and within-window variable storing difference between citing and citation dates
       window_weeks = int(round(window_years * 52.18)) + 3
       cit["lag_w"] = (cit["citing_gdate"] - cit["cited_gdate"]).astype("Int64")
       cit = cit[(cit["lag_w"] >= 0) & (cit["lag_w"] <= window_weeks)].copy()
# else, if GDATE does not exist, we use GYEAR instead
   else:
       cit = cit.dropna(subset=["citing_year", "cited_year"]).copy()
       cit["lag_y"] = (cit["citing_year"] - cit["cited_year"]).astype("Int64")
       # with lag 0 at the year level to count citations occurring in the same year
       cit = cit[(cit["lag_y"] >= 0) & (cit["lag_y"] <= window_years)].copy()


   # forward citations count (distinct nunique citing patents, avoids redundant citations within same citing patent)
   fwd_counts = cit.groupby("CITED")["CITING"].nunique().rename("fwd_cites_5y")


   # self-citations within window (boolean + self counts column)
   cit["is_self_cite"] = (
       cit["cited_assignee"].notna()
       & cit["citing_assignee"].notna()
       & (cit["cited_assignee"] == cit["citing_assignee"])
   )
   self_counts = cit.loc[cit["is_self_cite"]].groupby("CITED")["CITING"].nunique().rename("self_cites_5y")


   # merge counts back to focal table (renaming + matching counts with patent IDs + no within-window citations become integer 0)
   focal = focal.rename(columns={"PATENT": "patent_id", "GYEAR": "grant_year"})
   focal = focal.merge(fwd_counts, left_on="patent_id", right_index=True, how="left")
   focal = focal.merge(self_counts, left_on="patent_id", right_index=True, how="left")
   focal["fwd_cites_5y"] = focal["fwd_cites_5y"].fillna(0).astype(int)
   focal["self_cites_5y"] = focal["self_cites_5y"].fillna(0).astype(int)


   return focal

In [ ]:
# panel-building for event-study as row per year (variables aggregation per grant_year + x==0 as booleans average to shares in pd + sorted by year + reset)
def build_yearly_panel(focal_patents: pd.DataFrame) -> pd.DataFrame:
   g = focal_patents.groupby("grant_year", as_index=False).agg(
       patents_n=("patent_id", "count"),
       mean_fwd_cites_5y=("fwd_cites_5y", "mean"),
       median_fwd_cites_5y=("fwd_cites_5y", "median"),
       mean_self_cites_5y=("self_cites_5y", "mean"),
       share_zero_cites_5y=("fwd_cites_5y", lambda x: (x == 0).mean())
   )
   return g.sort_values("grant_year").reset_index(drop=True)


# time series plot with event vertical line (quantity, quality, 0-citations graphs)
# used at the beginning to check for mistakes / surprising data features.
def plot_time_series(yearly: pd.DataFrame, policy_year: int = 1981) -> None:
   plt.figure()
   plt.plot(yearly["grant_year"], yearly["patents_n"])
   plt.axvline(policy_year, linestyle="--")
   plt.title("Patent grants per year (focal sample)")
   plt.xlabel("Grant year"); plt.ylabel("Number of patents")
   plt.show()


   plt.figure()
   plt.plot(yearly["grant_year"], yearly["mean_fwd_cites_5y"])
   plt.axvline(policy_year, linestyle="--")
   plt.title("Mean forward citations within 5 years (by grant year)")
   plt.xlabel("Grant year"); plt.ylabel("Mean 5-year forward citations")
   plt.show()


   plt.figure()
   plt.plot(yearly["grant_year"], yearly["share_zero_cites_5y"])
   plt.axvline(policy_year, linestyle="--")
   plt.title("Share of patents with zero forward citations within 5 years")
   plt.xlabel("Grant year"); plt.ylabel("Share zero-cited (5y)")
   plt.show()

In [ ]:
# -----------------------------
# RUN + DIAGNOSTICS : data & focal window summary + plots display
# -----------------------------
patents = load_patents_dynamic(PATENTS_CSV_PATH)
citations = load_citations(CITES_CSV_PATH)


print("Patents years:", int(patents["GYEAR"].min()), "to", int(patents["GYEAR"].max()))
print("Citations rows:", len(citations))
print("Has GDATE:", "GDATE" in patents.columns)


focal = construct_forward_citations_fixed_window(
   patents=patents,
   citations=citations,
   focal_start_year=1975,
   focal_end_year=1994,
   window_years=5
)


yearly = build_yearly_panel(focal)

# checks focal group coherence with patent dataset : year range [1975,1994], patents, 10-last years prints.
print("Focal year range:", int(yearly["grant_year"].min()), "to", int(yearly["grant_year"].max()))
print("Focal patents rows:", len(focal))
print("Yearly tail:")
print(yearly.tail(10))


plot_time_series(yearly, policy_year=1981)

In [ ]:
####################################
####################################
# ============================================================
# Section 2 — Assignee-year Event Study / DiD
# Outcomes: (1) log patent counts, (2) mean 5y forward citations
# Contrast: US corporations vs foreign corporations (assignee type)
# ============================================================


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# If needed:
!pip -q install linearmodels
from linearmodels.panel import PanelOLS


POLICY_YEAR = 1981

In [ ]:
def normalize_asscode(asscode: pd.Series) -> pd.Series:
   """
   Normalize ASSCODE so 'part interest' codes (e.g., 12, 13) map to base codes (2, 3).
   PatentsView documentation notes that a leading '1' indicates part interest. :contentReference[oaicite:4]{index=4}
   """
   x = pd.to_numeric(asscode, errors="coerce")
   base = np.where(x >= 10, x % 10, x)
   return pd.Series(base, index=asscode.index).astype("Int64")




def build_assignee_year_panel(focal: pd.DataFrame) -> pd.DataFrame:
   """
   Build assignee-year outcomes from patent-level focal data.


   Returns a panel with:
     - patents_n
     - fwd_cites_5y_sum
     - mean_fwd_cites_5y
     - treated (US corp=1, foreign corp=0)
   """
   df = focal.copy()
   df["asscode_base"] = normalize_asscode(df["ASSCODE"])
   df = df[df["ASSIGNEE"].notna()].copy()


   # Keep only corporate assignees: US corp (2) and foreign corp (3)
   df = df[df["asscode_base"].isin([2, 3])].copy()
   df["treated"] = (df["asscode_base"] == 2).astype(int)


   panel = df.groupby(["ASSIGNEE", "grant_year"], as_index=False).agg(
       patents_n=("patent_id", "count"),
       fwd_cites_5y_sum=("fwd_cites_5y", "sum"),
       self_cites_5y_sum=("self_cites_5y", "sum"),
   )


   panel["mean_fwd_cites_5y"] = panel["fwd_cites_5y_sum"] / panel["patents_n"]
   panel["mean_self_cites_5y"] = panel["self_cites_5y_sum"] / panel["patents_n"]


   # Attach treatment at assignee-year by merging the modal asscode_base within assignee
   ass_treat = df.groupby("ASSIGNEE")["treated"].mean().rename("treated_assignee").reset_index()
   # treated_assignee should be ~0 or ~1; threshold at 0.5
   ass_treat["treated_assignee"] = (ass_treat["treated_assignee"] >= 0.5).astype(int)
   panel = panel.merge(ass_treat, on="ASSIGNEE", how="left")


   # Event time
   panel["event_time"] = panel["grant_year"] - POLICY_YEAR


   return panel




def make_event_dummies(panel: pd.DataFrame, min_k: int = -6, max_k: int = 10, omit_k: int = -1) -> (pd.DataFrame, list):
   """
   Create treated x event_time dummies for event-study, omitting one pre-period (default -1).
   """
   ks = [k for k in range(min_k, max_k + 1) if k != omit_k]
   for k in ks:
       panel[f"evt_{k}"] = ((panel["event_time"] == k) & (panel["treated_assignee"] == 1)).astype(int)
   return panel, ks




def fit_event_study(panel: pd.DataFrame, y_col: str, min_k=-6, max_k=10, omit_k=-1, weight_col="patents_n"):
   """
   Two-way FE event study:
     y_{a,t} = sum_k beta_k * 1[treat_a=1]*1[event_time=k] + alpha_a + delta_t + eps_{a,t}
   Weighted by patents_n by default.
   """
   df = panel.copy()
   df = df[(df["event_time"] >= min_k) & (df["event_time"] <= max_k)].copy()
   df, ks = make_event_dummies(df, min_k=min_k, max_k=max_k, omit_k=omit_k)


   df = df.set_index(["ASSIGNEE", "grant_year"]).sort_index()


   X = df[[f"evt_{k}" for k in ks]]
   y = df[y_col]


   weights = df[weight_col] if weight_col in df.columns else None


   mod = PanelOLS(y, X, entity_effects=True, time_effects=True, weights=weights)
   res = mod.fit(cov_type="clustered", cluster_entity=True)


   return res, ks




def plot_event_coefs(res, ks, title: str):
   params = res.params
   se = res.std_errors


   beta = np.array([params.get(f"evt_{k}", np.nan) for k in ks], dtype=float)
   s = np.array([se.get(f"evt_{k}", np.nan) for k in ks], dtype=float)


   plt.figure()
   plt.axhline(0, linestyle="--")
   plt.plot(ks, beta)
   plt.fill_between(ks, beta - 1.96*s, beta + 1.96*s, alpha=0.2)
   plt.axvline(0, linestyle="--")
   plt.title(title)
   plt.xlabel("Event time (years relative to 1981)")
   plt.ylabel("Coefficient (treated × event time)")
   plt.show()

In [ ]:
# -----------------------------
# RUN Section 2
# -----------------------------


panel = build_assignee_year_panel(focal)


# Outcomes
panel["log_patents_n"] = np.log1p(panel["patents_n"])
panel["log_mean_fwd_cites_5y"] = np.log1p(panel["mean_fwd_cites_5y"])


print("Panel rows:", len(panel))
print(panel.head())


# Event study on patenting intensity (quantity)
res_q, ks_q = fit_event_study(panel, y_col="log_patents_n", min_k=-6, max_k=10, omit_k=-1)
print(res_q.summary)
plot_event_coefs(res_q, ks_q, "Event study: log(1+patents) — US corp vs foreign corp")


# Event study on citation intensity (quality)
res_qual, ks_qual = fit_event_study(panel, y_col="log_mean_fwd_cites_5y", min_k=-6, max_k=10, omit_k=-1)
print(res_qual.summary)
plot_event_coefs(res_qual, ks_qual, "Event study: log(1+mean 5y forward citations) — US corp vs foreign corp")

In [ ]:
####################################
####################################
# ============================================================
# Section 2 bis - completion with high/low R&D exposure
#
# ============================================================


# to erase if redundant at the end of the project when aggregating codes
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# probably not redundant
from linearmodels.panel import PanelOLS


POLICY_YEAR = 1981


#
def normalize_asscode(asscode: pd.Series) -> pd.Series:
   """
   Map ASSCODE part-interest codes (>=10) to base codes using modulo 10.
   Example: 12 -> 2, 13 -> 3.
   """
   x = pd.to_numeric(asscode, errors="coerce")
   base = np.where(x >= 10, x % 10, x)
   return pd.Series(base, index=asscode.index).astype("Int64")




def build_uscorp_panel(focal: pd.DataFrame) -> pd.DataFrame:
   """
   Build an assignee-year panel for US corporate assignees only.


   Outputs:
     patents_n, mean_fwd_cites_5y, event_time
   """
   df = focal.copy()
   df["asscode_base"] = normalize_asscode(df["ASSCODE"])
   df = df[df["ASSIGNEE"].notna()].copy()


   # US corporations only
   df = df[df["asscode_base"] == 2].copy()


   panel = df.groupby(["ASSIGNEE", "grant_year"], as_index=False).agg(
       patents_n=("patent_id", "count"),
       fwd_cites_5y_sum=("fwd_cites_5y", "sum"),
   )
   panel["mean_fwd_cites_5y"] = panel["fwd_cites_5y_sum"] / panel["patents_n"]
   panel["event_time"] = panel["grant_year"] - POLICY_YEAR
   return panel




def assign_exposure_groups(panel: pd.DataFrame, pre_start=1975, pre_end=1980, q=0.75) -> pd.DataFrame:
   """
   Define 'high exposure' assignees as those above the q-quantile
   of total pre-policy patents (pre_start..pre_end).
   """
   pre = panel[(panel["grant_year"] >= pre_start) & (panel["grant_year"] <= pre_end)].copy()
   pre_tot = pre.groupby("ASSIGNEE")["patents_n"].sum().rename("pre_patents").reset_index()


   cutoff = pre_tot["pre_patents"].quantile(q)
   pre_tot["high_exposure"] = (pre_tot["pre_patents"] >= cutoff).astype(int)


   out = panel.merge(pre_tot[["ASSIGNEE", "high_exposure", "pre_patents"]], on="ASSIGNEE", how="inner")
   return out




def make_event_dummies(df: pd.DataFrame, group_col: str, min_k=-6, max_k=10, omit_k=-1):
   """
   Create event-study dummies: group × event_time.
   """
   ks = [k for k in range(min_k, max_k + 1) if k != omit_k]
   for k in ks:
       df[f"evt_{k}"] = ((df["event_time"] == k) & (df[group_col] == 1)).astype(int)
   return df, ks




def fit_event_study(df: pd.DataFrame, y_col: str, ks, weight_col="patents_n"):
   """
   Two-way FE event study:
     y_{a,t} = sum_k beta_k * 1[group_a=1]*1[event=k] + alpha_a + delta_t + eps
   Weighted by patents_n by default.
   """
   dfi = df.set_index(["ASSIGNEE", "grant_year"]).sort_index()
   X = dfi[[f"evt_{k}" for k in ks]]
   y = dfi[y_col]
   w = dfi[weight_col]
   res = PanelOLS(y, X, entity_effects=True, time_effects=True, weights=w).fit(
       cov_type="clustered", cluster_entity=True
   )
   return res




def plot_event(res, ks, title):
   beta = np.array([res.params.get(f"evt_{k}", np.nan) for k in ks], float)
   se = np.array([res.std_errors.get(f"evt_{k}", np.nan) for k in ks], float)


   plt.figure()
   plt.axhline(0, linestyle="--")
   plt.axvline(0, linestyle="--")
   plt.plot(ks, beta)
   plt.fill_between(ks, beta - 1.96 * se, beta + 1.96 * se, alpha=0.2)
   plt.title(title)
   plt.xlabel("Event time (years relative to 1981)")
   plt.ylabel("Coefficient (high exposure × event time)")
   plt.show()

In [ ]:
# -------------------------
# RUN: High vs Low exposure
# -------------------------
us_panel = build_uscorp_panel(focal)
us_panel = assign_exposure_groups(us_panel, pre_start=1975, pre_end=1980, q=0.75)


# Outcomes
us_panel["log_patents_n"] = np.log1p(us_panel["patents_n"])
us_panel["log_mean_fwd"] = np.log1p(us_panel["mean_fwd_cites_5y"])
us_panel["asinh_mean_fwd"] = np.arcsinh(us_panel["mean_fwd_cites_5y"])  # robustness


# Event window
us_panel = us_panel[(us_panel["event_time"] >= -6) & (us_panel["event_time"] <= 10)].copy()
us_panel, ks = make_event_dummies(us_panel, group_col="high_exposure", min_k=-6, max_k=10, omit_k=-1)


# Quantity
res_q = fit_event_study(us_panel, "log_patents_n", ks)
print(res_q.summary)
plot_event(res_q, ks, "US corporations: log(1+patents) — high vs low pre-1981 exposure")


# Quality
res_qual = fit_event_study(us_panel, "asinh_mean_fwd", ks)
print(res_qual.summary)
plot_event(res_qual, ks, "US corporations: asinh(mean 5y forward citations) — high vs low pre-1981 exposure")

In [ ]:
####################################
####################################
# ============================================================
# Utilities for project paper - (tables etc.)
# ///
# ============================================================


# =========================
# CONFIG (edit to match your notebook objects)
# =========================
import numpy as np
import pandas as pd


# If you already created these earlier, point to them here:
FOCAL_PATENTS_DF_NAME = "focal_patents"   # patent-level df with fwd_cites_5y
ASSIGNEE_YEAR_PANEL_DF_NAME = "us_panel"  # assignee-year panel (US corporations sample) used for regressions


# Key columns expected:
# focal_patents: PATENT, GYEAR (grant year), ASSIGNEE, CAT/SUBCAT, fwd_cites_5y (int), self_cites_5y (optional)
# us_panel: ASSIGNEE, grant_year, patents_n, mean_fwd_cites_5y, high_exposure, event_time


# Output folder (created if missing)
OUTPUT_DIR = "paper_outputs"




# =========================
# UTILITIES
# =========================
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


def safe_get_df(name: str) -> pd.DataFrame:
   """
   Fetch a dataframe from the current notebook namespace by name.


   Parameters
   ----------
   name : str
       Variable name (string) of the dataframe.


   Returns
   -------
   pd.DataFrame
       The dataframe.


   Raises
   ------
   KeyError
       If the dataframe is not found in globals().
   """
   if name not in globals():
       raise KeyError(f"DataFrame `{name}` not found. Update CONFIG or create it earlier in the notebook.")
   return globals()[name].copy()


def winsorize_series(x: pd.Series, p_low=0.01, p_high=0.99) -> pd.Series:
   """
   Winsorize a numeric series to limit the influence of extreme outliers.


   Parameters
   ----------
   x : pd.Series
       Numeric series.
   p_low : float
       Lower tail probability.
   p_high : float
       Upper tail probability.


   Returns
   -------
   pd.Series
       Winsorized series.
   """
   lo, hi = x.quantile(p_low), x.quantile(p_high)
   return x.clip(lower=lo, upper=hi)

In [ ]:
# =========================
# (A) Summary statistics for paper (patent-level and assignee-year)
# =========================

def make_summary_stats_patent_level(focal_patents: pd.DataFrame) -> pd.DataFrame:
   """
   Create patent-level summary statistics for the focal sample.


   Returns a compact table suitable for the Data section.
   """
   df = focal_patents.copy()


   # Required checks
   required = ["grant_year", "fwd_cites_5y"]
   for c in required:
       if c not in df.columns:
           raise ValueError(f"Missing required column: {c}")


   out = pd.DataFrame({
       "N patents": [len(df)],
       "Years (min)": [df["grant_year"].min()],
       "Years (max)": [df["grant_year"].max()],
       "Mean 5y forward cites": [df["fwd_cites_5y"].mean()],
       "Median 5y forward cites": [df["fwd_cites_5y"].median()],
       "Share zero 5y forward cites": [(df["fwd_cites_5y"] == 0).mean()],
   })


   if "self_cites_5y" in df.columns:
       out["Mean 5y self cites"] = df["self_cites_5y"].mean()


   return out


def make_yearly_trends_table(focal_patents: pd.DataFrame) -> pd.DataFrame:
   """
   Yearly trends used for Figure 1/2 (patent counts + citations).
   """
   df = focal_patents.copy()
   g = df.groupby("grant_year").agg(
       patents_n=("PATENT", "count") if "PATENT" in df.columns else ("fwd_cites_5y","size"),
       mean_fwd_cites_5y=("fwd_cites_5y", "mean"),
       median_fwd_cites_5y=("fwd_cites_5y", "median"),
       share_zero_cites_5y=("fwd_cites_5y", lambda x: (x == 0).mean()),
   ).reset_index().rename(columns={"grant_year":"grant_year"}) # Renaming redundant if groupby already uses 'grant_year'
   return g


focal_patents = safe_get_df(FOCAL_PATENTS_DF_NAME)
summary_pat = make_summary_stats_patent_level(focal_patents)
yearly_trends = make_yearly_trends_table(focal_patents)


display(summary_pat)
display(yearly_trends.head(10))


summary_pat.to_csv(os.path.join(OUTPUT_DIR, "summary_patent_level.csv"), index=False)
yearly_trends.to_csv(os.path.join(OUTPUT_DIR, "yearly_trends.csv"), index=False)
print("Saved summary tables to:", OUTPUT_DIR)

In [ ]:
# =========================
# (B) Pre/post descriptive table around 1981
# =========================
def pre_post_table(focal_patents: pd.DataFrame, policy_year=1981, pre_years=(1975, 1980), post_years=(1982, 1987)) -> pd.DataFrame:
   """
   Produce a simple pre/post comparison table (means) for paper narrative.
   """
   df = focal_patents.copy()
   df = df[(df["grant_year"] >= pre_years[0]) & (df["grant_year"] <= post_years[1])]


   df["period"] = np.where(
       (df["grant_year"] >= pre_years[0]) & (df["grant_year"] <= pre_years[1]), "Pre",
       np.where((df["grant_year"] >= post_years[0]) & (df["grant_year"] <= post_years[1]), "Post", "Other")
   )
   df = df[df["period"].isin(["Pre","Post"])]


   tab = df.groupby("period").agg(
       patents_n=("fwd_cites_5y","size"),
       mean_fwd_cites_5y=("fwd_cites_5y","mean"),
       median_fwd_cites_5y=("fwd_cites_5y","median"),
       share_zero_cites_5y=("fwd_cites_5y", lambda x: (x==0).mean()),
   )


   if "self_cites_5y" in df.columns:
       tab["mean_self_cites_5y"] = df.groupby("period")["self_cites_5y"].mean()


   return tab.reset_index()


pp = pre_post_table(focal_patents)
display(pp)
pp.to_csv(os.path.join(OUTPUT_DIR, "pre_post_table.csv"), index=False)

In [ ]:
# =========================
# (C) Event-study coefficient table export (from a linearmodels PanelOLS fit)
# =========================
def extract_event_study_params(results, prefix="evt_") -> pd.DataFrame:
   """
   Extract event-study coefficients from a linearmodels PanelOLS results object.


   Parameters
   ----------
   results : linearmodels.panel.results.PanelEffectsResults
       Fitted regression results.
   prefix : str
       Parameter prefix used for event-time dummies.


   Returns
   -------
   pd.DataFrame
       Columns: event_time, coef, se, ci_low, ci_high, pvalue
   """
   params = results.params
   ses = results.std_errors
   pvals = results.pvalues


   rows = []
   for k in params.index:
       if str(k).startswith(prefix):
           # parse evt_-6 -> -6
           et = int(str(k).replace(prefix,""))
           b = float(params.loc[k])
           se = float(ses.loc[k])
           rows.append({
               "event_time": et,
               "coef": b,
               "se": se,
               "pvalue": float(pvals.loc[k]),
               "ci_low": b - 1.96*se,
               "ci_high": b + 1.96*se,
           })


   out = pd.DataFrame(rows).sort_values("event_time").reset_index(drop=True)
   return out


# Example usage:
# est_table = extract_event_study_params(res_patents)  # replace with your fitted results object
# est_table.to_csv(os.path.join(OUTPUT_DIR, "event_study_patents.csv"), index=False)

In [ ]:
####################################
####################################
# ============================================================
# Predictive Model
# ///
# ============================================================


# =========================
# (D) Predictive step: classify highly-cited patents (top decile)
# =========================
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report


def build_prediction_dataset(focal_patents: pd.DataFrame,
                            start_year=1975, end_year=1994,
                            sample_n=250_000,
                            random_state=42) -> pd.DataFrame:

   df = focal_patents.copy()
   df = df[(df["grant_year"] >= start_year) & (df["grant_year"] <= end_year)].copy()


   # Cleaning in numeric
   for col in ["CLAIMS", "APPYEAR"]:
       if col in df.columns:
           df[col] = pd.to_numeric(df[col], errors="coerce")


   # Define target within year (simple; robust variant: within year x SUBCAT)
# computes 90th percentile for each grant-year row of the table + if in its year’s p90 the patent is indicated highly cited (HIGH_CITED)
   df["p90_year"] = df.groupby("grant_year")["fwd_cites_5y"].transform(lambda x: x.quantile(0.90))
   df["HIGH_CITED"] = (df["fwd_cites_5y"] >= df["p90_year"]).astype(int)


   # Sampling
   if sample_n is not None and len(df) > sample_n:
       df = df.sample(sample_n, random_state=random_state).copy()


   # Feature set : ensures every interesting feature is there, incl. HIGH_CITED
   keep_cols = ["grant_year","APPYEAR","CLAIMS","COUNTRY","POSTATE","ASSCODE","CAT","SUBCAT","NCLASS","HIGH_CITED"]
   keep_cols = [c for c in keep_cols if c in df.columns]
   df = df[keep_cols].copy()


   # Fill for missing numerics/categoricals with median/UNK
   for c in df.columns:
       if c == "HIGH_CITED":
           continue
       if pd.api.types.is_numeric_dtype(df[c]):
           df[c] = df[c].fillna(df[c].median())
       else:
           df[c] = df[c].fillna("UNK")


   return df
# I kept both in the end because one of them is not defined but I could not recall which one
model_def=build_prediction_dataset(focal, sample_n=250_000)
model_df = build_prediction_dataset(focal_patents, sample_n=250_000)
print(model_df.shape)
display(model_df.head())




# =========================
# (D) Predictive step: classify highly-cited patents (top decile)
# =========================


from sklearn.preprocessing import StandardScaler


# arbitrary split train/test samples
train_df = model_df[model_df["grant_year"] <= 1988].copy()
test_df  = model_df[model_df["grant_year"] >= 1989].copy()
# extracts target values
y_train = train_df["HIGH_CITED"].values
y_test  = test_df["HIGH_CITED"].values


X_train = train_df.drop(columns=["HIGH_CITED"])
X_test  = test_df.drop(columns=["HIGH_CITED"])


# Define feature types numerical/categorical
num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]


# ignores unknown
preprocess = ColumnTransformer(
   transformers=[
       ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
       ("num", StandardScaler(), num_cols), # Add StandardScaler for numerical features
   ]
)


clf = LogisticRegression(
   max_iter=1000,  # max_iter to increase if ConvergenceWarning
   n_jobs=None,
   C=1.0,
   solver="lbfgs",
   class_weight='balanced' # Add class_weight to handle imbalance of selecting 90th percentile
)


pipe = Pipeline(steps=[("prep", preprocess), ("clf", clf)])


pipe.fit(X_train, y_train)


# probability of class HIGH_CITED=1
p_test = pipe.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, p_test)
print("Test AUC:", auc)


pred = (p_test >= 0.5).astype(int)
print(classification_report(y_test, pred, digits=3))